# Metadata Upgrade Status Analysis
Analyze upgrade success rates across projects using the metadata_upgrade_status_prod table

In [9]:
import pandas as pd
import altair as alt
from zombie_squirrel import custom
from aind_data_access_api.document_db import MetadataDbClient

In [10]:
df = custom("metadata_upgrade_status_prod")
print(f"RDS table shape: {df.shape}")
print(f"\nRDS table columns: {df.columns.tolist()}")

client = MetadataDbClient(
    host="api.allenneuraldynamics.org",
    version="v1",
)

extra_columns = {
    "_id": 1,
    "data_description.data_level": 1,
    "data_description.project_name": 1,
    "name": 1,
}

print("\nRetrieving project_name and data_level from DocDB...")
all_records = client.retrieve_docdb_records(
    filter_query={},
    projection={"_id": 1},
    limit=0,
)
all_ids = [record["_id"] for record in all_records]
print(f"Found {len(all_ids)} total records in DocDB")

batch_size = 100
records = []
for start_idx in range(0, len(all_ids), batch_size):
    end_idx = start_idx + batch_size
    batch_ids = all_ids[start_idx:end_idx]
    filter_query = {"_id": {"$in": batch_ids}}
    batch_records = client.retrieve_docdb_records(
        filter_query=filter_query,
        projection=extra_columns,
        limit=0,
    )
    records.extend(batch_records)

for i, record in enumerate(records):
    data_description = record.get("data_description", {})
    if data_description:
        record["data_level"] = data_description.get("data_level", None)
        record["project_name"] = data_description.get("project_name", None)
        record.pop("data_description")
    records[i] = record

extra_df = pd.DataFrame(records)
print(f"Retrieved {len(extra_df)} records from DocDB")

print("\nMerging tables...")
df = df.merge(extra_df, how="left", left_on="v1_id", right_on="_id")
print(f"Merged table shape: {df.shape}")
print(f"Merged table columns: {df.columns.tolist()}")

RDS table shape: (80767, 5)

RDS table columns: ['v1_id', 'v2_id', 'upgrader_version', 'last_modified', 'status']

Retrieving project_name and data_level from DocDB...
Found 80781 total records in DocDB
Retrieved 80781 records from DocDB

Merging tables...
Merged table shape: (80767, 9)
Merged table columns: ['v1_id', 'v2_id', 'upgrader_version', 'last_modified', 'status', '_id', 'name', 'data_level', 'project_name']


In [11]:
df.head(10)

,v1_id,v2_id,upgrader_version,last_modified,status,_id,name,data_level,project_name
0,000099c0-eab2-4d83-bcdc-440954c1e60d,a82c2998-6e26-47de-baea-b12c425ba30f,0.8.0,2026-03-10T04:48:31.581Z,success,000099c0-eab2-4d83-bcdc-440954c1e60d,behavior_775510_2025-04-13_15-30-30_processed_...,derived,AIND Viral Genetic Tools
1,0002954e-88c0-4878-8417-241cc07e432c,7893efe0-2557-4f50-8169-11847acb9cad,0.8.0,2026-03-10T04:54:07.384Z,success,0002954e-88c0-4878-8417-241cc07e432c,behavior_792288_2025-07-22_08-38-59_processed_...,derived,Discovery-Neuromodulator circuit dynamics duri...
2,00054555-cfdc-490f-8d0e-c350da4156dc,70edd757-3b60-4282-9ef4-dcf73a42ef1e,0.8.0,2024-11-26T20:53:33.359904Z,success,00054555-cfdc-490f-8d0e-c350da4156dc,behavior_638573_2022-11-10_13-55-38,raw,Dynamic Routing
3,00075070-719b-4575-9c4c-2d81ec241db3,None,0.8.0,2024-11-26T20:26:30.729780Z,failed,00075070-719b-4575-9c4c-2d81ec241db3,behavior_676909_2023-11-03_09-16-17,raw,Dynamic Routing
4,0007a18c-e1c3-426d-af54-8d5a1aeab018,48819b80-b35e-4577-99c6-08bbe8163206,0.8.0,2026-03-10T04:55:23.467Z,success,0007a18c-e1c3-426d-af54-8d5a1aeab018,behavior_770803_2025-02-13_13-04-57,raw,Behavior Platform
5,0008d481-1376-4be9-92d0-c61d1785c7ad,None,0.8.0,2025-10-31T20:15:57.710Z,failed,0008d481-1376-4be9-92d0-c61d1785c7ad,HCR_732195-ROI2-cell15_2024-06-15_06-00-00,raw,None
6,00099710-d776-42c1-ac3a-bd45985cc2b6,None,0.8.0,2025-10-31T20:16:43.663Z,failed,00099710-d776-42c1-ac3a-bd45985cc2b6,HCR_772646-5a-4_2025-07-15_10-00-00,raw,None
7,000ab43e-4841-4bef-8e06-b8a99828cff3,48866135-6ac6-427b-9f84-706ddbaf3f51,0.8.0,2026-03-10T04:49:34.788Z,success,000ab43e-4841-4bef-8e06-b8a99828cff3,behavior_786237_2025-07-02_09-03-01,raw,Discovery-Neuromodulator circuit dynamics duri...
8,000ac49c-afa3-47bd-9064-e0ae5ecefbd4,None,0.8.0,2026-03-10T04:40:36.133Z,failed,000ac49c-afa3-47bd-9064-e0ae5ecefbd4,behavior_820117_2026-01-19_16-26-39,raw,Striatum
9,000ace99-03f8-49f3-a3c1-f3d57de83e3a,None,0.8.0,2025-10-31T20:16:24.162Z,failed,000ace99-03f8-49f3-a3c1-f3d57de83e3a,HCR_772643-4a-2_2025-02-26_10-00-00_registered...,derived,None


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80767 entries, 0 to 80766
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   v1_id             80767 non-null  object
 1   v2_id             45012 non-null  object
 2   upgrader_version  80767 non-null  object
 3   last_modified     80767 non-null  object
 4   status            80767 non-null  object
 5   _id               80767 non-null  object
 6   name              80767 non-null  object
 7   data_level        78343 non-null  object
 8   project_name      69592 non-null  object
dtypes: object(9)
memory usage: 5.5+ MB


In [13]:
df.describe(include='all')

,v1_id,v2_id,upgrader_version,last_modified,status,_id,name,data_level,project_name
count,80767,45012,80767,80767,80767,80767,80767,78343,69592
unique,80767,45012,1,74943,2,80767,80186,5,68
top,000099c0-eab2-4d83-bcdc-440954c1e60d,a82c2998-6e26-47de-baea-b12c425ba30f,0.8.0,2025-08-25T21:31:53.116Z,success,000099c0-eab2-4d83-bcdc-440954c1e60d,behavior_746346_2025-01-09_10-23-01_processed_...,raw,Dynamic Routing
freq,1,1,80767,66,45012,1,52,42458,11572


In [14]:
print("Status values:")
print(df['status'].value_counts())
print(f"\nProject names ({df['project_name'].nunique()} unique):") 
print(df['project_name'].value_counts().head(20))

Status values:
status
success    45012
failed     35755
Name: count, dtype: int64

Project names (68 unique):
project_name
Dynamic Routing                                                                                                                             11572
Behavior Platform                                                                                                                            9682
Genetic Perturbation Platform                                                                                                                7568
Discovery-Neuromodulator circuit dynamics during foraging - Subproject 3 Fiber Photometry Recordings of NM Release During Behavior           6421
Cognitive flexibility in patch foraging                                                                                                      5188
Learning mFISH-V1omFISH                                                                                                                      3956
D

In [ ]:
success_by_project = df.groupby('project_name').size().reset_index(name='total_records')
success_by_project = success_by_project.sort_values('total_records', ascending=False)
print("Records by project:")
print(success_by_project)

In [15]:
success_pct_by_project = df.groupby('project_name').apply(
    lambda x: (x['status'] == 'success').sum() / len(x) * 100
).reset_index(name='success_percentage')
success_pct_by_project = success_pct_by_project.sort_values('success_percentage', ascending=False)
print("Success percentage by project:")
print(success_pct_by_project)

Success percentage by project:
                                         project_name  success_percentage
67                                      omFISHSstMeso               100.0
13           CelltypesTransgenicCharacterizationGCaMP               100.0
66             XPG Core Imaging - Methods Development               100.0
58  Thalamus - Project 4 Frontal cortical dynamics...               100.0
51                                     OpenScopeVismo               100.0
..                                                ...                 ...
53                             Ophys Platform - SLAP2                 0.0
7                          Behavior and Motor Control                 0.0
44                                  Molecular Anatomy                 0.0
47                             Neurobiology of Action                 0.0
34                               High and Low Control                 0.0

[68 rows x 2 columns]


In [16]:
chart_success = alt.Chart(success_pct_by_project).mark_bar(color='steelblue').encode(
    x=alt.X('project_name:N', sort='-y', title='Project Name'),
    y=alt.Y('success_percentage:Q', title='Success Percentage (%)', scale=alt.Scale(domain=[0, 100])),
    tooltip=['project_name:N', alt.Tooltip('success_percentage:Q', format='.2f')]
).properties(
    width=1000,
    height=500,
    title='Upgrade Success Rate by Project'
)
chart_success.display()

alt.Chart(...)

In [17]:
success_detail = df.groupby(['project_name', 'status']).size().reset_index(name='count')
chart_stacked = alt.Chart(success_detail).mark_bar().encode(
    x=alt.X('project_name:N', sort='-y', title='Project Name'),
    y=alt.Y('count:Q', title='Count'),
    color=alt.Color('status:N', title='Status'),
    tooltip=['project_name:N', 'status:N', 'count:Q']
).properties(
    width=1000,
    height=500,
    title='Upgrade Status Breakdown by Project'
).interactive()
chart_stacked.display()

alt.Chart(...)

In [18]:
print(f"Total records: {len(df)}")
print(f"Total projects analyzed: {df['project_name'].nunique()}")
print(f"\nOverall status distribution:")
print(df['status'].value_counts())
print(f"\nOverall success rate: {(df['status'] == 'success').sum() / len(df) * 100:.2f}%")

Total records: 80767
Total projects analyzed: 68

Overall status distribution:
status
success    45012
failed     35755
Name: count, dtype: int64

Overall success rate: 55.73%


In [ ]:
if status_column and len(success_pct_by_project) > 0:
    chart_success = alt.Chart(success_pct_by_project).mark_bar(color='steelblue').encode(
        x=alt.X('project_name:N', sort='-y', title='Project Name'),
        y=alt.Y('success_percentage:Q', title='Success Percentage (%)', scale=alt.Scale(domain=[0, 100])),
        tooltip=['project_name:N', alt.Tooltip('success_percentage:Q', format='.2f')]
    ).properties(
        width=800,
        height=400,
        title=f'Upgrade Success Rate by Project ({status_column})'
    )
    chart_success.display()

In [ ]:
if status_column:
    success_detail = df.groupby(['project_name', status_column]).size().reset_index(name='count')
    chart_stacked = alt.Chart(success_detail).mark_bar().encode(
        x=alt.X('project_name:N', sort='-y', title='Project Name'),
        y=alt.Y('count:Q', title='Count'),
        color=alt.Color(f'{status_column}:N', title=status_column),
        tooltip=['project_name:N', status_column, 'count:Q']
    ).properties(
        width=800,
        height=400,
        title=f'Upgrade Status Breakdown by Project'
    ).interactive()
    chart_stacked.display()

## Summary Statistics

In [ ]:
print(f"Total records: {len(df)}")
print(f"Total projects: {df['project_name'].nunique()}")
if status_column:
    print(f"\nOverall status distribution:")
    print(df[status_column].value_counts())
    print(f"\nMost common status: {df[status_column].mode()[0]}")